In [ ]:
# NOTEBOOK NAME
# SimplePPIplotsHorizontal.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

# for projecting radar coordinates to lat and lon
from pyproj import Geod

# mapping things
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

# for adding lat/lon gridlines on plots
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# # SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *

# for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

In [ ]:
# PPI DATA VERSION
# PPI DATA VERSION
# HORIZONTAL CROSS SECTION PLOTTING (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# CHOOSE YOUR RADAR

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald
# RadarIDno = '23' 

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 12
RadarDay   = 1
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '12:00'
LoopEndTime   = '12:00'

# CHOOSE YOUR ELEVATION ANGLE
# ElevationAngle = 23  # [degrees] options are    0.5   0.8   1.4   2.4
                       #                          3.5   4.7   6.0   7.8
                       #                           10    13    17    23   and   32

# calculate the ground range when beam altitude reaches 20 km for a max ground radius of consideration 
RangeOf20km = 20 / (np.tan(np.deg2rad(ElevationAngle)))

# choose how far you want the extremes of the box in the plot from the radar
BoxRange = np.minimum(150.0, RangeOf20km ) # [km] set that maximum ground range as a limit to the plot, or 150 km as a max value


# elevation angle choice follow-on
# make sure that the chosen elevation angle is valide
AllElevationAngles = np.array([0.5, 0.8, 1.4, 2.4, 3.5, 4.7, 6.0, 7.8, 10.0, 13.0, 17.0, 23.0, 32.0])
assert ElevationAngle in AllElevationAngles, 'Please choose one of the available elevation angles: ' + \
                                            '0.5, 0.8, 1.4, 2.4, 3.5, 4.7, 6.0, 7.8, 10.0, 13.0, 17.0, 23.0, 32.0'

# CHOOSE YOUR VARIABLE
# Var = 'Z'
# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [RhoHV]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
...
# [V]         'corrected_velocity'

# VARIABLES NOT YET INCLUDED IN THE CODE
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
...
# [AzSh]      'azshear'



# USER CHOICE FOLLOW-ON SECTION

# ra
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton (Brisbane)'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marburg'
elif (RadarIDno == '19'):
    RadarSiteName = 'Cairns'
elif (RadarIDno == '8'):
    RadarSiteName = 'Gympie'
elif (RadarIDno == '24'):
    RadarSiteName = 'Bowen'
elif (RadarIDno == '23'):
    RadarSiteName = 'Gladstone'
elif (RadarIDno == '74'):
    RadarSiteName = 'Greenvale'
elif (RadarIDno == '98'):
    RadarSiteName = 'Taroom'
elif (RadarIDno == '108'):
    RadarSiteName = 'Towoomba'
elif (RadarIDno == '78'):
    RadarSiteName = 'Weipa'
elif (RadarIDno == '72'):
    RadarSiteName = 'Emerald'
else:
    RadarSiteName = 'Site ' + RadarIDno

# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# Longitude shift correction for known coordinate errors
if RadarSiteName == 'Townsville':
    LonShift = 146.5505 - (-19.4195)
else:
    LonShift = 0.0



# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -10 # [dBZ]
    VarMaxVal =   65 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 10

elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -5.0
    VarColourBar_max = 5.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 1.0

elif (Var == 'RhoHV'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.8
    VarColourBar_max = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 0.05
    
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = -2  # [deg/ km]
    VarMaxVal = 2 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -2
    VarColourBar_max = 2
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 0.2

elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.0
    VarColourBar_max = 30.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 5.0

elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max =  30.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 5.0

else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

    

# remember these plots are horizontal cross sections
PlotType = 'HorzPPI'
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)
    # xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
    #                         RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')


    NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi' + '/'
    NetCDFstorageFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi.nc'
    NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile
    
    
    # try to load in the netcdf file and if it doesn't work, just keep going
    try:
        RadarXR = xr.open_dataset(NetCDFstoragePath, decode_timedelta = False) # add the decode_timedelta to shut up a warning
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # ALL IN ONE BLOCK

    # calculate the effective radius of the earth given standard refraction
    RadiusEarth = 6371000 # [m]
    RadiusEarthEff = RadiusEarth * (4/3)
    
    # load in a "geodesic calculator"
    geod = Geod(ellps='WGS84')
    
    # read in the PPI coordinates from the xarray dataframe
    Ranges  = RadarXR['range'].values
    AziDegs = RadarXR['azimuth'].values
    EleDegs = RadarXR['elevation'].values
    
    # read in the radar site location from the xarray dataframe
    SiteLat = float(RadarXR.latitude)
    SiteLon = float(RadarXR.longitude)
    SiteAlt = float(RadarXR.altitude)
    
    # convert angles to radians
    AziRads = np.deg2rad(AziDegs)[:,None]
    EleRads = np.deg2rad(EleDegs)[:,None]
    
    # I AM UNCERTAIN OF THIS TRIGONOMETRIC FORMULA
    
    # Distance from Centre of the Earth using 4/3 Earth-radius model
    DistsFromCOE = np.sqrt(    (Ranges**2)  +  (RadiusEarthEff**2)  +  (2.0 * Ranges * RadiusEarthEff * np.sin(EleRads))     )
    # Convert to altitude by subtracting out the earth and adding in the site elevation
    Alts = (DistsFromCOE - RadiusEarthEff) + SiteAlt
    
    # MAYBE THIS FORMULA SHOULD ALSO TAKE INTO ACCOUNT EARTH CURVATURE?
    
    # calculate range in distance along the surface of Earth
    GroundRanges = Ranges * np.cos(EleRads)  # distance along surface [m]
    # the last line adds a dimension of 1 at the end for some reason that I remove here:
    GroundRanges = np.squeeze(GroundRanges)
    
    # retrieve number of ranges and gates in the ppi file
    NumRays, NumGates = GroundRanges.shape
    
    # for i in range(NumRays):
        
    # Broadcast (repeat) radar site lon/lat and azimuth to match GroundRanges shape
    SiteLonRep = np.full_like(GroundRanges, SiteLon, dtype=float)
    SiteLatRep = np.full_like(GroundRanges, SiteLat, dtype=float)
    
    AziDegsRep= np.repeat(AziDegs[:, None], NumGates, axis=1)
    # AziDegsRep = np.full_like(GroundRanges, AziDegs, dtype=float)
    
    Lons, Lats, _ = geod.fwd(SiteLonRep, SiteLatRep, AziDegsRep, GroundRanges)
    
    
    # turn the calculated latitudes and longitudes and altitudes for each range gate into xarray elements 
    
    LatsData = xr.DataArray(Lats, dims=('time', 'range'), name='gate_latitude',
        attrs={
            'long_name': 'latitude of radar gates',
            'units': 'degrees_north'})
    
    LonsData = xr.DataArray(Lons + LonShift, dims=('time', 'range'), name='gate_longitude',
        attrs={
            'long_name': 'longitude of radar gates',
            'units': 'degrees_east'})

    
    AltsData = xr.DataArray(Alts, dims=('time', 'range'), name='gate_altitude',
        attrs={
            'long_name': 'altitude of radar gates',
            'units': 'm',
            'standard_name': 'altitude'})
    
    
    # and add them as variables to your radar data
    RadarXR['gate_latitude']  = LatsData
    RadarXR['gate_longitude'] = LonsData
    RadarXR['gate_altitude']  = AltsData

    fig, ax = plt.subplots(figsize=(10, 8), 
                           subplot_kw={'projection': ccrs.PlateCarree()})

    # gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    
    # TERRAIN SHADING USING LOCAL GEBCO DEM
    lon_min, lon_max = float(RadarXR.gate_longitude.min()), float(RadarXR.gate_longitude.max())

    lat_min, lat_max = float(RadarXR.gate_latitude.min()), float(RadarXR.gate_latitude.max())

    try:
        gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
        
        # print(f"Attempting to load DEM with bounds:")
        # print(f"  Lon: {lon_min} to {lon_max}")
        # print(f"  Lat: {lat_min} to {lat_max}")
        
        dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)
        
        if dem_da is None:
            raise ValueError('GEBCO DEM returned None')
        
        dem_lon = dem_da.lon.values
        dem_lat = dem_da.lat.values
        dem_data = dem_da.values
  
        # print(f"DEM data range: {np.nanmin(dem_data)} to {np.nanmax(dem_data)} m")
        # print(f"NaN count: {np.isnan(dem_data).sum()}")

        # Make 2D lon/lat grids if necessary
        if dem_lon.ndim == 1 and dem_lat.ndim == 1:
            dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
        else:
            dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

        # Ensure we have some valid data
        valid = np.isfinite(dem_data)
        if not np.any(valid):
            raise ValueError('DEM has no finite values in this domain')

        colours = [
            '#dde4e8',  # 0: pale blue-grey (ocean, < 0 m)
        
            '#c4dec2',  # 1: 0–200 m, pale green
            '#e4edc9',  # 2: 200–400 m, greenish-yellow
            '#f3f0cf',  # 3: 400–600 m, pale yellow-beige
            '#e9d7bd',  # 4: 600–800 m, light tan
            '#ddc4aa',  # 5: 800–1000 m, tan
            '#cfb194',  # 6: 1000–1200 m, light brown
            '#b58f6e',  # 7: > 1200 m, darker brown
        ]
        
        bounds = [
            -1000.0,  # ocean below 0
            0.0,      # 0–200
            200.0,    # 200–400
            400.0,    # 400–600
            600.0,    # 600–800
            800.0,    # 800–1000
            1000.0,   # 1000–1200
            1200.0,   # > 1200
            5000.0,
        ]
        
        cmap_elev = ListedColormap(colours)
        norm = BoundaryNorm(bounds, len(colours), clip=True)
        
        # Plot as semi‑transparent background
        elev_plot = ax.pcolormesh(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            cmap=cmap_elev,
            norm=norm,
            alpha=1.0,
            transform=ccrs.PlateCarree(),
            # zorder=2,
        )

        # Draw 0 m contour as an accurate coastline
        coast_contour = ax.contour(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            levels=[0.0],
            colors='black',
            linewidths=0.5,
            transform=ccrs.PlateCarree(),
            zorder=15,  # above radar and topo
        )

        # Draw 400 m contour
        coast_contour = ax.contour(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            levels=[400.0],
            colors='black',
            linewidths=0.3,
            transform=ccrs.PlateCarree(),
            zorder=15,  # above radar and topo
        )

        # print('Terrain shading (GEBCO, discrete bands) + coastline loaded successfully')

    except Exception as e:
        print(f'Terrain shading failed: {e}')
        # print('Falling back to simple land/ocean shading')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
        ax.add_feature(cfeature.LAND, facecolor='#E8E8E8', alpha=0.3, zorder=2)

    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)
    # END EXPERIMENTAL TOPO SHADING SECTION
    # END EXPERIMENTAL TOPO SHADING SECTION
    # END EXPERIMENTAL TOPO SHADING SECTION

    # ACTUAL PLOTTING RIGHT HERE
    # ACTUAL PLOTTING RIGHT HERE
    # ACTUAL PLOTTING RIGHT HERE, select the "time coordinates" for the given elevation angle
    EleAngleMask = (RadarXR.elevation == ElevationAngle) # make a mask to only plot the one elevation angle
    Lats = RadarXR.gate_latitude[EleAngleMask]
    Lons = RadarXR.gate_longitude[EleAngleMask]
    PlotVar = RadarXR[VarNameLong][EleAngleMask]

    PlotVar = PlotVar.where(PlotVar != VarFillValue, np.nan) # get rid of the fill values in the variable you are plotting
    
    GridViewer = ax.pcolormesh( Lons, Lats, PlotVar, cmap=VarColourBar, norm=VarColourBar_norm, shading='auto', transform=ccrs.PlateCarree() )

    cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
    cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
    cbar.set_ticks(np.arange(VarMinVal, VarMaxVal, VarTickSpacing))  # tick every 10 dBZ

    # plot a star for the location of the radar on the map
    ax.plot(float(RadarXR.longitude) + LonShift, float(RadarXR.latitude),
            marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=20)
    ax.plot(float(RadarXR.longitude) + LonShift, float(RadarXR.latitude),
            marker='*', color='white', markersize=4, transform=ccrs.PlateCarree(), zorder=21)


    # Add MINOR gridlines (tenth degrees) - thin
    gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
    gl_minor.xlocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees
    gl_minor.ylocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees

    # Add MID LEVEL gridlines (half degrees) - standard width with labels
    gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
    gl_mid.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
    gl_mid.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

    # Add MAJOR gridlines (full degrees) - thick with labels
    gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
    gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    
    # Format labels
    gl_mid.xformatter = LONGITUDE_FORMATTER
    gl_mid.yformatter = LATITUDE_FORMATTER
    
    gl_major.xformatter = LONGITUDE_FORMATTER
    gl_major.yformatter = LATITUDE_FORMATTER
    
    # Remove labels from top and right
    gl_mid.top_labels = False
    gl_mid.right_labels = False
    gl_mid.bottom_labels = True
    gl_mid.left_labels = True
    
    gl_major.top_labels = False
    gl_major.right_labels = False
    gl_major.bottom_labels = True
    gl_major.left_labels = True

    # calculate how many degrees this range is (different for latitude vs longitude)
    RadiusEarth = 6371 # [km]
    kmPerDegLat = (RadiusEarth * 2 * np.pi) /360
    kmPerDegLon = kmPerDegLat * np.cos( np.deg2rad(RadarXR.latitude))
    
    BoxRangeDegLat = BoxRange / kmPerDegLat
    BoxRangeDegLon = BoxRange / kmPerDegLon

    plt.xlim([float(RadarXR.longitude) + LonShift - BoxRangeDegLon, float(RadarXR.longitude) + LonShift + BoxRangeDegLon])
    plt.ylim([float(RadarXR.latitude)   - BoxRangeDegLat,  float(RadarXR.latitude)  + BoxRangeDegLat])
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    plt.title(VarName + ' for ' + RadarSiteName + ' Radar\nfor the ' + str(ElevationAngle) + '° Elevation Angle\non ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')

    SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
    SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + \
                 PlotType + str(int(ElevationAngle*10)) + 'deg.png'
    
    SavePath = SaveFolder + SaveFile
    
    if not Path(SaveFolder).exists():
        print('doing')
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
    # plt.close()

In [ ]:
# CHOOSE YOUR RADAR

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald
RadarIDno = '23' 

# CHOOSE YOUR ELEVATION ANGLE
ElevationAngle = 0.8  # [degrees] options are    0.5   0.8   1.4   2.4
                       #                         3.5   4.7   6.0   7.8
                       #                          10    13    17    23   and   32

# CHOOSE YOUR VARIABLE
Var = 'Z'

In [ ]:
# Townsville Data mistakenly has Lon = Lat

Lat: -19.4195
Lon Should Be: +146.5505

Just Push Everything by (146.5505 - (-19.4195))

In [ ]:
from scipy.stats import gaussian_kde

# Define your fill values
reflectivity_fill_values = [-30.0, -999.0]  # Replace with actual values
phase_fill_values = [-5.0, 7.50038148136845]  # Replace with actual values

# Extract the data
reflectivity = RadarXR['corrected_reflectivity'].values.flatten()
phase = RadarXR['corrected_specific_differential_phase'].values.flatten()

# Create a mask for valid data (not NaN and not fill values)
valid_mask = ~np.isnan(reflectivity) & ~np.isnan(phase)

for fill_val in reflectivity_fill_values:
    valid_mask &= (reflectivity != fill_val)

for fill_val in phase_fill_values:
    valid_mask &= (phase != fill_val)

# Filter to valid points only
reflectivity_valid = reflectivity[valid_mask]
phase_valid = phase[valid_mask]
fig, ax = plt.subplots(figsize=(10, 8))

hexbin = ax.hexbin(reflectivity_valid, phase_valid, gridsize=30, cmap='viridis', mincnt=1)

ax.set_xlabel('Corrected Reflectivity (dBZ)', fontsize=12)
ax.set_ylabel('Corrected Specific Differential Phase (°/km)', fontsize=12)
ax.set_title('Reflectivity vs Differential Phase Density', fontsize=14)

cbar = plt.colorbar(hexbin, ax=ax)
cbar.set_label('Count', fontsize=11)

plt.tight_layout()
plt.show()
print(f"Valid points: {len(reflectivity_valid)}")

In [ ]:
fill_value = -5.0
PlotVarArray = np.array(PlotVar)

# Method 1: Using boolean indexing
valid_values = PlotVarArray[(~np.isnan(PlotVarArray)) & (PlotVarArray != fill_value)]

In [ ]:
unique_values, counts = np.unique(np.array(PlotVar), return_counts=True)

for value, count in zip(unique_values, counts):
    print(f"{value}: {count} occurrence{'s' if count > 1 else ''}")

In [ ]:
# GIF MAKER
# PPI VERSION
# FOR Horizontal Cross Sections

# PlotVar  = 'corrected_reflectivity'
PlotType = 'HorzPPI'

SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
                
# LOADING IMAGES
files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named

images = [Image.open(os.path.join(SavedFolder, f))
    for f in files
    if f.endswith(PlotType + str(Altitude) + 'm.png')]



GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + VarNameLong + '_' + str(ElevationAngle) + 'deg.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile


# make a folder to store the new GIF in if one does not exist already
if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(GIFsavePath, save_all=True, append_images=images[1:], duration=200, loop=0)          
                                                                   # ms per frame       0 = loop forever
print('Saved GIF for ' + RadarFileDate)